# Silver Append-Only Template

Bronze → Silver insert-only loader for event and transaction contracts. Per the temporal-modeling and event-and-transaction ADRs, append-only tables do not carry SCD2 fields and are never updated. Corrections arrive as new rows referencing the corrected row via `corrects_*_uid`.

Reads the manifest path as a parameter and adapts to any contract of `contractKind: event` or `contractKind: transaction` with `appendOnly.enabled: true`.

Generated by `scripts/generation/generate-fabric-notebooks.py`. Edit the generator, never this file.

In [ ]:
# PARAMETERS
manifest_path = "targets/fabric/manifests/pc/policy/policy-lifecycle-event.fabric.yaml"
load_mode = "incremental"          # incremental | full | catchup
as_of_datetime = None              # ISO-8601 string upper bound for catchup
bronze_prefix_override = None      # optional bronze.table prefix override
run_optimize = False               # OPTIMIZE after append (rare; usually skipped)
optimize_min_rows = 1_000_000

In [ ]:
# Imports, Spark session config, and timezone.
#
# V-Order and Optimize Write are enabled at session level per
# `targets/fabric/conventions.md` §7. The notebook runs in UTC; the SCD2
# merge stamps `valid_from_datetime` from `current_timestamp()`, so the
# session timezone must be UTC for the windows to line up across
# environments.

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import yaml
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DecimalType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

In [ ]:
# Load the manifest and run a structural sanity check.
#
# Drift between the manifest and the canonical contract is enforced at
# generation time by `scripts/validation/validate-fabric-manifests.py`. The
# checks here are runtime guards against the manifest having been edited or
# truncated by hand, which is forbidden by the metadata-driven posture in
# `planning-mds/FABRIC_IMPLEMENTATION_PLAN.md` §2.

manifest_file = Path(manifest_path)
if not manifest_file.is_absolute():
    manifest_file = Path.cwd() / manifest_file

with manifest_file.open("r", encoding="utf-8") as handle:
    manifest = yaml.safe_load(handle)

if not isinstance(manifest, dict):
    raise ValueError(f"Manifest at {manifest_file} did not parse as a mapping")

contract = manifest.get("contract") or {}
fabric = manifest.get("fabric") or {}
columns = list(fabric.get("columns") or [])
if not columns:
    raise ValueError("Manifest carries no fabric.columns")

contract_id = contract.get("id") or ""
contract_kind = contract.get("contractKind") or ""
lakehouse = fabric.get("lakehouse") or ""
schema = fabric.get("schema") or ""
table_block = fabric.get("table") or {}
table_name = table_block.get("name") or ""
fq_table = f"{lakehouse}.{schema}.{table_name}"

bronze_block = fabric.get("bronze") or {}
bronze_table = bronze_block.get("table") or ""
incremental_column = bronze_block.get("incrementalColumn") or "_ingested_at"
expected_columns = list(bronze_block.get("expectedColumns") or [])
if bronze_prefix_override and bronze_table:
    bronze_table = bronze_prefix_override + "." + bronze_table.split(".", 1)[-1]

quality_rules = list(fabric.get("qualityRules") or [])
delta_props = ((table_block.get("delta") or {}).get("tableProperties") or {})
partitioned_by = list((table_block.get("delta") or {}).get("partitionedBy") or [])
zorder_by = list((table_block.get("delta") or {}).get("zorderBy") or [])

print(
    f"Loaded manifest for {contract_id} ({contract_kind}) -> {fq_table}; "
    f"{len(columns)} columns, {len(quality_rules)} quality rule(s), "
    f"bronze={bronze_table or '(unset)'}"
)

In [ ]:
# Shared helpers: Spark type lookup, column projection, hash builder.

_PRIMITIVE_TYPES = {
    "STRING": StringType(),
    "INT": IntegerType(),
    "BIGINT": LongType(),
    "BOOLEAN": BooleanType(),
    "DATE": DateType(),
    "TIMESTAMP": TimestampType(),
}


def spark_type_for(spark_type_name: str):
    if spark_type_name in _PRIMITIVE_TYPES:
        return _PRIMITIVE_TYPES[spark_type_name]
    if spark_type_name.startswith("DECIMAL("):
        inner = spark_type_name[len("DECIMAL(") : -1]
        precision_str, scale_str = inner.split(",")
        return DecimalType(int(precision_str.strip()), int(scale_str.strip()))
    raise ValueError(f"Unsupported Spark type: {spark_type_name}")


def manifest_struct(columns_meta):
    """Return a StructType matching the manifest's column order."""
    fields = []
    for col in columns_meta:
        fields.append(
            StructField(
                col["name"],
                spark_type_for(col["sparkType"]),
                bool(col.get("nullable", True)),
            )
        )
    return StructType(fields)


def project_to_manifest(df, columns_meta):
    """Cast Bronze columns to the manifest's declared Spark types and drop
    columns the canonical contract does not carry. Missing required columns
    raise; missing optional columns are filled with NULL of the manifest type.
    """
    selected = []
    bronze_cols = set(df.columns)
    for col in columns_meta:
        name = col["name"]
        spark_type = spark_type_for(col["sparkType"])
        nullable = bool(col.get("nullable", True))
        if name in bronze_cols:
            selected.append(F.col(name).cast(spark_type).alias(name))
        else:
            if not nullable:
                raise ValueError(
                    f"Bronze missing required column {name!r}; manifest declares NOT NULL"
                )
            selected.append(F.lit(None).cast(spark_type).alias(name))
    return df.select(*selected)


def change_hash(columns_meta, exclude_names):
    """Build a SHA-256 hash expression over the columns not excluded from
    change detection. NULL is encoded distinctly so a transition NULL->value
    or value->NULL counts as a change.
    """
    parts = []
    for col in columns_meta:
        name = col["name"]
        if name in exclude_names:
            continue
        parts.append(
            F.when(F.col(name).isNull(), F.lit("\u0000NULL\u0000"))
            .otherwise(F.col(name).cast("string"))
        )
        parts.append(F.lit("\u0001"))
    if not parts:
        return F.lit("")
    concat = F.concat(*parts)
    return F.sha2(concat, 256)

In [ ]:
# Kind guard. Append-only template handles event and transaction contracts.

if contract_kind not in ("event", "transaction"):
    raise ValueError(
        f"silver-append-template requires contractKind in {{'event','transaction'}}, "
        f"got {contract_kind!r}. Use silver-scd2-merge-template for entities."
    )

append_block = fabric.get("appendOnly") or {}
append_enabled = bool(append_block.get("enabled", False))
if not append_enabled:
    raise ValueError(
        "Manifest has appendOnly.enabled=false; this template requires append-only mode."
    )

scd2_block = fabric.get("scd2") or {}
scd2_enabled = bool(scd2_block.get("enabled", False))
if scd2_enabled:
    raise ValueError(
        "Manifest has scd2.enabled=true alongside appendOnly.enabled=true; "
        "modes are mutually exclusive."
    )

In [ ]:
# Read Bronze with the manifest's incremental predicate.
#
# `load_mode == 'incremental'` filters Bronze on `incremental_column`
# greater than the maximum value already in Silver (or the parameter
# override `as_of_datetime`). `load_mode == 'full'` reads everything.
# `load_mode == 'catchup'` reads everything between a hand-supplied
# window via `as_of_datetime` (treated as the upper bound).
#
# Bronze schemas are allowed to carry extra columns; `project_to_manifest`
# strips them after the read.

if not bronze_table:
    raise ValueError("Manifest has no fabric.bronze.table; cannot read source")

bronze_full = spark.read.table(bronze_table)
incremental_present = incremental_column in bronze_full.columns

if load_mode == "full":
    bronze_filtered = bronze_full
elif load_mode == "incremental":
    high_water = None
    if as_of_datetime:
        high_water = as_of_datetime
    elif spark.catalog.tableExists(fq_table) and incremental_present:
        existing = spark.read.table(fq_table)
        if incremental_column in existing.columns:
            row = existing.agg(F.max(incremental_column).alias("hw")).collect()
            high_water = row[0]["hw"] if row else None
    if high_water is not None and incremental_present:
        bronze_filtered = bronze_full.filter(
            F.col(incremental_column) > F.lit(high_water)
        )
    else:
        bronze_filtered = bronze_full
elif load_mode == "catchup":
    if not as_of_datetime:
        raise ValueError("load_mode='catchup' requires as_of_datetime upper bound")
    if incremental_present:
        bronze_filtered = bronze_full.filter(
            F.col(incremental_column) <= F.lit(as_of_datetime)
        )
    else:
        bronze_filtered = bronze_full
else:
    raise ValueError(f"Unknown load_mode: {load_mode!r}")

bronze_df = project_to_manifest(bronze_filtered, columns)
bronze_count = bronze_df.count()
print(f"Bronze read: {bronze_count} row(s) from {bronze_table} ({load_mode})")

In [ ]:
# Pre-write quality assertions.
#
# Severity-`error` rules of types `not_null`, `unique`, `expression`, and
# `currency_pair` run against the Bronze dataframe before any Silver write.
# Failing pre-assertions abort the run; the merge does not proceed. Severity
# `warning` and `info` are recorded in the run summary without aborting.
#
# `accepted_values` (codeset) checks happen post-write because they need the
# loaded codeset table; see the post-write section.

assertion_results = []


def record_assertion(rule_id, rule_type, severity, passed, detail=""):
    assertion_results.append(
        {
            "rule": rule_id,
            "type": rule_type,
            "severity": severity,
            "phase": "pre" if severity != "post" else "post",
            "passed": bool(passed),
            "detail": detail,
        }
    )


def _eval_rule(df, rule):
    rule_id = rule.get("id") or "(unnamed)"
    rule_type = rule.get("type")
    severity = rule.get("severity") or "error"
    if rule_type == "not_null":
        column_name = rule.get("column")
        if column_name not in df.columns:
            return rule_id, severity, False, f"column {column_name!r} not in dataframe"
        bad = df.filter(F.col(column_name).isNull()).count()
        return rule_id, severity, bad == 0, f"{bad} null row(s) in {column_name}"
    if rule_type == "unique":
        keys = rule.get("keyColumns") or []
        if not keys:
            return rule_id, severity, False, "unique rule missing keyColumns"
        scoped = df
        if rule.get("filter"):
            scoped = scoped.filter(rule["filter"])
        dup = (
            scoped.groupBy(*keys)
            .count()
            .filter("count > 1")
            .count()
        )
        return rule_id, severity, dup == 0, f"{dup} duplicate key group(s)"
    if rule_type == "expression":
        expr = rule.get("expression") or "TRUE"
        bad = df.filter(f"NOT ({expr})").count()
        return rule_id, severity, bad == 0, f"{bad} row(s) violate expression"
    if rule_type == "currency_pair":
        amount = rule.get("amountColumn")
        currency = rule.get("currencyColumn")
        if not amount or not currency:
            return rule_id, severity, False, "currency_pair missing columns"
        bad = df.filter(
            F.col(amount).isNotNull() & F.col(currency).isNull()
        ).count()
        return (
            rule_id,
            severity,
            bad == 0,
            f"{bad} row(s) with {amount} but no {currency}",
        )
    if rule_type == "accepted_values":
        # Deferred to post-write; needs the codeset table.
        return rule_id, severity, True, "deferred to post-write"
    return rule_id, severity, True, f"unknown type {rule_type!r}; skipped"


error_failures = []
for rule in quality_rules:
    rule_type = rule.get("type") or ""
    severity = rule.get("severity") or "error"
    if rule_type == "accepted_values":
        continue
    rule_id, sev, passed, detail = _eval_rule(bronze_df, rule)
    record_assertion(rule_id, rule_type, sev, passed, detail)
    if sev == "error" and not passed:
        error_failures.append(f"{rule_id}: {detail}")

if error_failures:
    msg = "Pre-write assertions failed:\n  - " + "\n  - ".join(error_failures)
    raise RuntimeError(msg)
print(f"Pre-write assertions: {len(assertion_results)} evaluated, 0 errors")

In [ ]:
# Append-only insert body.
#
# Per `targets/fabric/conventions.md` §5, append-only contracts have four
# row outcomes:
#
#   - correction_indicator=FALSE, no row in Silver with the same *_uid:
#       insert.
#   - correction_indicator=FALSE, row in Silver with the same *_uid exists:
#       no-op (idempotent re-run).
#   - correction_indicator=TRUE, corrects_*_uid references a row that
#       exists in Silver: insert the correction row; the corrected row is
#       not updated or deleted.
#   - correction_indicator=TRUE, corrects_*_uid references a row that does
#       not exist in Silver: pre-write assertion fails (severity error),
#       run aborts.

correction_field = append_block.get("correctionIndicator")
corrects_ref_field = append_block.get("correctsRefField")

primary_key_col = next(
    (c for c in columns if c.get("primaryKey")), None
)
if primary_key_col is None:
    raise ValueError("Manifest has no column flagged primaryKey=true")
primary_key = primary_key_col["name"]

# Ensure target exists with the manifest schema.
if not spark.catalog.tableExists(fq_table):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {lakehouse}.{schema}")
    create_cols = []
    for col in columns:
        not_null = "" if col.get("nullable", True) else " NOT NULL"
        create_cols.append(f"  `{col['name']}` {col['sparkType']}{not_null}")
    column_clause = ",\n".join(create_cols)
    partition_clause = ""
    if partitioned_by:
        partition_clause = f"\nPARTITIONED BY ({', '.join(partitioned_by)})"
    tblprops = []
    for key in sorted(delta_props.keys()):
        val = delta_props[key]
        rendered = "true" if isinstance(val, bool) and val else (
            "false" if isinstance(val, bool) else str(val)
        )
        tblprops.append(f"'{key}' = '{rendered}'")
    tblprops_clause = ""
    if tblprops:
        tblprops_clause = "\nTBLPROPERTIES (\n  " + ",\n  ".join(tblprops) + "\n)"
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {fq_table} (\n{column_clause}\n) "
        f"USING DELTA{partition_clause}{tblprops_clause}"
    )

silver_keys = (
    spark.read.table(fq_table)
    .select(F.col(primary_key).alias("__pk"))
    .dropDuplicates()
)
src = bronze_df.alias("src")
keys = silver_keys.alias("tgt")

# Validate correction references before insert.
extra_failures = []
if correction_field and corrects_ref_field and corrects_ref_field in bronze_df.columns:
    correction_rows = bronze_df.filter(F.col(correction_field) == F.lit(True))
    if correction_rows.count() > 0:
        # corrects_*_uid must reference an existing primary key (in Silver
        # or in the same Bronze batch).
        existing_keys = (
            spark.read.table(fq_table).select(
                F.col(primary_key).alias("__ref_key")
            )
            .union(
                bronze_df.select(F.col(primary_key).alias("__ref_key"))
            )
            .dropDuplicates()
        )
        bad_refs = (
            correction_rows.select(
                F.col(corrects_ref_field).alias("__ref_key")
            )
            .filter(F.col("__ref_key").isNotNull())
            .join(existing_keys, on="__ref_key", how="left_anti")
        )
        bad_count = bad_refs.count()
        passed = bad_count == 0
        record_assertion(
            "correction_reference_resolves",
            "expression",
            "error",
            passed,
            f"{bad_count} correction row(s) reference unknown {primary_key}",
        )
        if not passed:
            extra_failures.append(
                f"correction_reference_resolves: {bad_count} row(s)"
            )
if extra_failures:
    raise RuntimeError(
        "Pre-write correction-reference assertions failed:\n  - "
        + "\n  - ".join(extra_failures)
    )

# Idempotent insert: drop bronze rows whose primary key already exists in
# Silver. Correction rows always have a fresh *_uid (the correction is a
# new fact); a duplicate *_uid in bronze is a true re-run, not a correction.
new_rows = src.join(
    keys, F.col(f"src.{primary_key}") == F.col("tgt.__pk"), "left_anti"
)
rows_read = bronze_count
rows_inserted = new_rows.count()
rows_superseded = 0
rows_soft_deleted = 0
rows_corrected = 0
if correction_field and correction_field in bronze_df.columns:
    rows_corrected = (
        new_rows.filter(F.col(correction_field) == F.lit(True)).count()
    )

if rows_inserted > 0:
    new_rows.write.format("delta").mode("append").saveAsTable(fq_table)

print(
    f"Append complete: read={rows_read}, inserted={rows_inserted}, "
    f"corrections={rows_corrected}"
)

In [ ]:
# Post-write validation.
#
# Codeset reference checks (`accepted_values`) and the SCD2-window /
# current-row-uniqueness checks run after the merge. Failures are recorded
# in the run summary and mark the run failed but do not roll back the
# write; the merge is idempotent and a re-run after the fix produces a
# clean state.

silver_df = spark.read.table(fq_table)
post_failures = []

# accepted_values checks (codeset reference resolution).
for rule in quality_rules:
    if rule.get("type") != "accepted_values":
        continue
    column_name = rule.get("column")
    codeset_table = rule.get("codesetTable")
    codeset_field = rule.get("codesetField") or "code_value"
    rule_id = rule.get("id") or "(unnamed)"
    severity = rule.get("severity") or "error"
    if not column_name or not codeset_table:
        record_assertion(rule_id, "accepted_values", severity, False, "missing config")
        post_failures.append(f"{rule_id}: missing config")
        continue
    if not spark.catalog.tableExists(codeset_table):
        detail = f"codeset table {codeset_table} not found"
        record_assertion(rule_id, "accepted_values", severity, False, detail)
        if severity == "error":
            post_failures.append(f"{rule_id}: {detail}")
        continue
    codeset_df = spark.read.table(codeset_table)
    if "is_current_indicator" in codeset_df.columns:
        codeset_df = codeset_df.filter(F.col("is_current_indicator") == F.lit(True))
    valid_codes = codeset_df.select(F.col(codeset_field).alias("code"))
    candidate = silver_df.filter(F.col(column_name).isNotNull()).select(
        F.col(column_name).alias("code")
    )
    bad = (
        candidate.join(valid_codes, "code", "left_anti").count()
    )
    passed = bad == 0
    detail = f"{bad} row(s) with {column_name} not in {codeset_table}"
    record_assertion(rule_id, "accepted_values", severity, passed, detail)
    if severity == "error" and not passed:
        post_failures.append(f"{rule_id}: {detail}")

# SCD2 / current-row checks for entity and codeset materializations.
if scd2_enabled:
    is_current_field = scd2_block.get("isCurrent") or "is_current_indicator"
    valid_from_field = scd2_block.get("validFrom") or "valid_from_datetime"
    valid_to_field = scd2_block.get("validTo") or "valid_to_datetime"
    natural_key = scd2_block.get("naturalKey") or []
    if natural_key:
        dup_current = (
            silver_df.filter(F.col(is_current_field) == F.lit(True))
            .groupBy(*natural_key)
            .count()
            .filter("count > 1")
            .count()
        )
        passed = dup_current == 0
        record_assertion(
            "single_current_row_per_key_post",
            "unique",
            "error",
            passed,
            f"{dup_current} key group(s) with multiple current rows",
        )
        if not passed:
            post_failures.append(
                f"single_current_row_per_key_post: {dup_current} duplicate group(s)"
            )

    bad_window = silver_df.filter(
        (F.col(valid_to_field).isNotNull())
        & (F.col(valid_to_field) <= F.col(valid_from_field))
    ).count()
    record_assertion(
        "scd2_window_consistent_post",
        "expression",
        "error",
        bad_window == 0,
        f"{bad_window} row(s) with valid_to <= valid_from",
    )
    if bad_window != 0:
        post_failures.append(
            f"scd2_window_consistent_post: {bad_window} row(s) violate window"
        )

if post_failures:
    print("Post-write assertions FAILED:")
    for line in post_failures:
        print(f"  - {line}")
else:
    print("Post-write assertions: all passed")

In [ ]:
# Optional OPTIMIZE / ZORDER advisory.
#
# `OPTIMIZE ... ZORDER` is a runtime command, so it lives in the notebook
# rather than DDL. The manifest's `zorderBy` hint runs only when the table
# already exists and `run_optimize` is true. Tables under 1M rows skip the
# rewrite to avoid pointless I/O; deployers tune the threshold via the
# notebook parameter.

if run_optimize and zorder_by:
    if spark.catalog.tableExists(fq_table):
        approx = silver_df.count()
        if approx >= optimize_min_rows:
            zorder_cols = ", ".join(zorder_by)
            spark.sql(f"OPTIMIZE {fq_table} ZORDER BY ({zorder_cols})")
            print(f"OPTIMIZE {fq_table} ZORDER BY ({zorder_cols}) issued")
        else:
            print(
                f"Skipping OPTIMIZE: {approx} row(s) below threshold "
                f"{optimize_min_rows}"
            )
    else:
        print(f"Skipping OPTIMIZE: {fq_table} does not exist yet")
else:
    print("OPTIMIZE skipped (run_optimize=False or no zorderBy hint)")

In [ ]:
# Run summary.
#
# A structured dictionary with per-rule outcomes and merge counters. The
# orchestrator (Fabric pipeline activity, REST caller, or CI runner) parses
# this to decide success/failure and to feed run telemetry. Print as JSON
# for log capture; do not raise here -- pre-write errors aborted the run
# already, and post-write errors are surfaced in the summary's `status`.

summary = {
    "contractId": contract_id,
    "contractKind": contract_kind,
    "table": fq_table,
    "loadMode": load_mode,
    "runDatetime": datetime.now(timezone.utc).isoformat(),
    "rowsRead": int(rows_read),
    "rowsInserted": int(rows_inserted),
    "rowsSuperseded": int(rows_superseded),
    "rowsSoftDeleted": int(rows_soft_deleted),
    "rowsCorrected": int(rows_corrected),
    "assertions": assertion_results,
    "status": "failed" if post_failures else "ok",
}
print(json.dumps(summary, indent=2))

if post_failures:
    raise RuntimeError(
        f"Run completed with {len(post_failures)} post-write assertion failure(s); "
        f"see summary above. Re-run after fix to clear."
    )